In [ ]:
import ast
import pandas as pd
import numpy as np

In [ ]:
import jax
from jax import jit
import jax.numpy as jnp
from jax.scipy.spatial.transform import Rotation

In [ ]:
IDX_Q = slice(0, 4)
IDX_P = slice(4, 7)
IDX_O = slice(7, 10)
IDX_V = slice(10, 13)
IDX_BATTERY = slice(13, 14)
IDX_TRIM = slice(14, 15)

MASS = 0.040
G_WORLD = jnp.array([0, 0, -9.8066])
I_TENSOR = jnp.diag(jnp.array([2e-4, 2e-4, 1e-4]))
I_INVERSE = jnp.diag(1 / (I_TENSOR.diagonal()))
DRAG = jnp.array([0.1,  0.4, 0.25])

MAX_THRUST = jnp.array([0.0, 0.0, 0.80])
MAX_TAIL_THRUST = jnp.array([0.0, 0.0, 0.125])

TAIL_MOMENT_ARM = 0.12

YAW_TIME_CONSTANT = 0.03

GYRO_SPRING_CONSTANT = jnp.array([0.01, 0.01, 0.0])
ANGULAR_DRAG = jnp.array([0.05, 0.025, 1e-4])
CORIOLIS_CONSTANT = 0.06

In [ ]:
@jit
def propagate(s, dt, commands, ground=False):
    # Position in world frame
    # Velocity in body frame

    pos_old = s[IDX_P]
    vel_old = s[IDX_V]
    omega_old = s[IDX_O]
    quat_old = Rotation.from_quat(s[IDX_Q])
    battery = s[IDX_BATTERY]
    trim_old = s[IDX_TRIM]

    thrust, pitch, yaw = commands

    # Quaternion
    dq = Rotation.from_rotvec(omega_old * dt)
    quat_new = quat_old * dq

    # Position
    vel_rotated = quat_old.apply(vel_old)
    pos_new = pos_old + vel_rotated * dt

    # Velocity
    gravity = quat_new.inv().apply(G_WORLD)
    thrust_vector = thrust * MAX_THRUST * battery + pitch * MAX_TAIL_THRUST * battery
    drag = DRAG * vel_old
    F_net_body = gravity + thrust_vector - drag
    F_net_world_frame = quat_new.apply(F_net_body)
    normal = jnp.where(ground, quat_new.inv().apply(jnp.array([0.0, 0, -F_net_world_frame[2]])), jnp.array([0.0, 0.0, 0.0]))
    F_net = F_net_body + normal
    acc_coriolis = jnp.cross(omega_old, vel_old)

    vel_new = vel_old + ((F_net / MASS) - acc_coriolis) * dt

    # Angular Velocity
    tau_roll = acc_coriolis[1] * MASS * CORIOLIS_CONSTANT
    tau_actuator_pitch = (MAX_TAIL_THRUST[2] * pitch * battery * TAIL_MOMENT_ARM).squeeze()
    tau_actuator_yaw = ((I_TENSOR[2, 2] / YAW_TIME_CONSTANT) * jnp.clip(yaw + trim_old, -1.0, 1.0) * battery).squeeze()
    tau_actuator = jnp.array([tau_roll, tau_actuator_pitch, tau_actuator_yaw])

    tau_gyro = GYRO_SPRING_CONSTANT * quat_old.as_euler(seq='XYZ', degrees=False)
    tau_damping = ANGULAR_DRAG * omega_old

    tau_net = tau_actuator - tau_gyro - tau_damping

    omega_cross_I_omega = jnp.cross(omega_old, I_TENSOR @ omega_old)
    d_omega = I_INVERSE @ (tau_net - omega_cross_I_omega)

    omega_new = omega_old + d_omega * dt

    # Battery
    battery_new = battery - (thrust * (dt / 360.))

    # New State
    s_new = jnp.concatenate([quat_new.as_quat(canonical=True), pos_new, omega_new, vel_new, battery_new, trim_old])

    return s_new

In [ ]:
df = pd.read_csv('./flight_recordings/recording.csv')
df['command'] = df['command'].apply(lambda x: np.array(ast.literal_eval(x)))
df['position'] = df['position'].apply(lambda x: np.array(ast.literal_eval(x)))

In [ ]:
df

In [ ]:
initial_state = jnp.array([          0,           0,   -0.095246,     0.99545,    0.010593,     -0.4701,           0,           0,           0,           0,           0,           0,           0,           0,           0])

In [ ]:
def run_simulation(_initial_state, timestamps, commands_seq):
    dts = jnp.diff(timestamps)
    commands_to_apply = commands_seq[:-1]

    def step_fn(state, inputs):
        dt, commands = inputs
        ground = state[IDX_P][2] < 0.075
        next_state = propagate(state, dt, commands, ground=ground)
        return next_state, next_state

    _, states_history = jax.lax.scan(step_fn, _initial_state, (dts, commands_to_apply))

    return jnp.vstack((_initial_state, states_history))

In [ ]:
def simulate_trajectory(_initial_state, _df):
    commands = jnp.array(_df['command'].tolist())
    timestamps = jnp.array(_df['timestamp'].tolist())
    states_out = run_simulation(_initial_state, timestamps, commands)
    return states_out

In [ ]:
sim_out = simulate_trajectory(initial_state, df)

In [ ]:
sim_out[:, IDX_P]